# 01. Data Foundation

Construye la capa base del proyecto:
 - Ingesta de datos historicos
 - Limpieza robusta
 - Validacion de calidad
 - Definicion correcta del target
 - Preparacion para feature engineering (sin leakage)
 - Exportacion dataset limpio

In [21]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

rawPath = Path("../data/raw/international_results.csv")
cleanPath = Path("../data/processed/clean_matches.csv")
fullPath = Path("../data/processed/full_matches.csv")

rawUrl = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("Raw path:", rawPath)
print("Clean path:", cleanPath)
print("Full path:", fullPath)


Raw path: ..\data\raw\international_results.csv
Clean path: ..\data\processed\clean_matches.csv
Full path: ..\data\processed\full_matches.csv


## 1. Carga del historico

In [22]:
if rawPath.exists():
    rawDf = pd.read_csv(rawPath)
    sourceUsed = str(rawPath)
else:
    rawDf = pd.read_csv(rawUrl)
    rawPath.parent.mkdir(parents=True, exist_ok=True)
    rawDf.to_csv(rawPath, index=False)
    sourceUsed = rawUrl

# Renombrar columnas (camelCase)
rawDf = rawDf.rename(
    columns={
        "home_team": "homeTeam",
        "away_team": "awayTeam",
        "home_score": "homeScore",
        "away_score": "awayScore",
    }
)

# Tipos
rawDf["date"] = pd.to_datetime(rawDf["date"])

# Orden temporal (CRITICO)
rawDf = rawDf.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Source used:", sourceUsed)
print("Shape rawDf:", rawDf.shape)
rawDf.head()

Source used: ..\data\raw\international_results.csv
Shape rawDf: (49071, 9)


,date,homeTeam,awayTeam,homeScore,awayScore,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


## 2. Limpieza y validacion de calidad

In [23]:
requiredColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "homeScore",
    "awayScore",
    "tournament",
    "city",
    "country",
    "neutral",
]

# Validar columnas
missingColumns = [col for col in requiredColumns if col not in rawDf.columns]
assert not missingColumns, f"Faltan columnas requeridas: {missingColumns}"

# Validar nulos
assert rawDf[requiredColumns].isnull().sum().sum() == 0, "Hay nulos en columnas criticas"

# Validar scores
assert (rawDf[["homeScore", "awayScore"]] < 0).sum().sum() == 0, "Hay marcadores negativos"


In [24]:
# Eliminar duplicados (IMPORTANTE)
initialShape = rawDf.shape
rawDf = rawDf.drop_duplicates().reset_index(drop=True)

print("Filas eliminadas por duplicados:", initialShape[0] - rawDf.shape[0])
print("Shape tras drop_duplicates:", rawDf.shape)

Filas eliminadas por duplicados: 0
Shape tras drop_duplicates: (49071, 9)


In [25]:
# Diagnosticar duplicados logicos antes de resolver conflictos
duplicateMatchKeys = rawDf.duplicated(subset=["date", "homeTeam", "awayTeam"]).sum()
print("Duplicados logicos detectados antes de limpieza:", duplicateMatchKeys)

Duplicados logicos detectados antes de limpieza: 1


In [26]:
duplicateDf = rawDf[
    rawDf.duplicated(subset=["date", "homeTeam", "awayTeam"], keep=False)
].sort_values(["date", "homeTeam", "awayTeam"])

duplicateDf.head(20)

,date,homeTeam,awayTeam,homeScore,awayScore,tournament,city,country,neutral
9637,1974-02-17,Tahiti,New Caledonia,2,1,Friendly,Papeete,Tahiti,False
9638,1974-02-17,Tahiti,New Caledonia,1,2,Friendly,Papeete,Tahiti,False


In [27]:
duplicateDf[[
    "date", "homeTeam", "awayTeam",
    "tournament", "city", "country",
    "homeScore", "awayScore"
]].head(20)

,date,homeTeam,awayTeam,tournament,city,country,homeScore,awayScore
9637,1974-02-17,Tahiti,New Caledonia,Friendly,Papeete,Tahiti,2,1
9638,1974-02-17,Tahiti,New Caledonia,Friendly,Papeete,Tahiti,1,2


Esto nos genera dos registros:

| date       | homeTeam | awayTeam      | score | city    |
| ---------- | -------- | ------------- | ----- | ------- |
| 1974-02-17 | Tahiti   | New Caledonia | 2–1   | Papeete |
| 1974-02-17 | Tahiti   | New Caledonia | 1–2   | Papeete |

Podemos interepretar que:

### Opción 1 — Dos partidos reales
- Mismo día
- Mismo lugar
- Mismos equipos

👉 Extremadamente improbable

### Opción 2 — Error en el dataset (más probable)

Uno de los registros está mal:

- score invertido
- duplicación errónea

Este tipo de inconsistencias es conocido en datasets históricos.

In [28]:
# Detectar posibles conflictos (mismo partido, distinto score)
conflictMask = rawDf.duplicated(
    subset=["date", "homeTeam", "awayTeam", "city"],
    keep=False
)

conflictsDf = rawDf[conflictMask].copy()

In [29]:
# Identificar conflictos reales (scores distintos)
conflictGroups = (
    conflictsDf
    .groupby(["date", "homeTeam", "awayTeam", "city"])
    .filter(lambda x: x[["homeScore", "awayScore"]].nunique().sum() > 2)
)

print("Conflictos detectados:", len(conflictGroups))

Conflictos detectados: 2


In [30]:
# Eliminar TODOS los registros conflictivos
if len(conflictGroups) > 0:
    conflictKeys = conflictGroups[["date", "homeTeam", "awayTeam", "city"]].drop_duplicates()

    rawDf = rawDf.merge(
        conflictKeys,
        on=["date", "homeTeam", "awayTeam", "city"],
        how="left",
        indicator=True
    )

    rawDf = rawDf[rawDf["_merge"] == "left_only"].drop(columns=["_merge"])

duplicateMatchKeys = rawDf.duplicated(subset=["date", "homeTeam", "awayTeam"]).sum()
assert duplicateMatchKeys == 0, "Persisten duplicados logicos tras limpiar conflictos"

print("Shape tras eliminar conflictos:", rawDf.shape)
print("Duplicados logicos tras limpieza:", duplicateMatchKeys)

Shape tras eliminar conflictos: (49069, 9)
Duplicados logicos tras limpieza: 0


## 4. Target correcto

In [31]:
rawDf["matchResult"] = np.select(
    [
        rawDf["homeScore"] > rawDf["awayScore"],
        rawDf["homeScore"] < rawDf["awayScore"],
    ],
    [
        "homeWin",
        "awayWin",
    ],
    default="draw",
)

rawDf["totalGoals"] = rawDf["homeScore"] + rawDf["awayScore"]

print(rawDf["matchResult"].value_counts(normalize=True).round(4))

matchResult
homeWin    0.4900
awayWin    0.2827
draw       0.2274
Name: proportion, dtype: float64


## 5. Tournament importance

In [32]:
def getTournamentWeight(tournament):
    tournament = tournament.lower()

    if "world cup" in tournament:
        return 1.0
    elif "qualifier" in tournament:
        return 0.8
    elif "euro" in tournament or "copa america" in tournament:
        return 0.7
    elif "nations league" in tournament:
        return 0.6
    elif "friendly" in tournament:
        return 0.3
    else:
        return 0.5

rawDf["tournamentWeight"] = rawDf["tournament"].apply(getTournamentWeight)


## 6. Dataset completo (para Elo)

In [38]:
fullDf = rawDf.copy()
fullDf.shape


(49069, 12)

In [39]:
fullPath.parent.mkdir(parents=True, exist_ok=True)
fullDf.to_csv(fullPath, index=False)
print("Data cleaning y feature engineering completados. Dataframe guardado en:", fullPath)

Data cleaning y feature engineering completados. Dataframe guardado en: ..\data\processed\full_matches.csv


## 7. Dataset para modelado

In [ ]:
analysisStartDate = pd.Timestamp("2000-01-01")

cleanDf = rawDf.loc[rawDf["date"] >= analysisStartDate].copy()
cleanDf = cleanDf.sort_values("date", kind="mergesort").reset_index(drop=True)

assert cleanDf.duplicated(subset=["date", "homeTeam", "awayTeam", "city"]).sum() == 0

cleanPath.parent.mkdir(parents=True, exist_ok=True)
cleanDf.to_csv(cleanPath, index=False)

print("Clean dataset shape:")
cleanDf.shape